# 🧬 NB00 — MLP Baseline (v3.3-ram · Multi-Cohort)
**Stage:** NB00 | **RAM strategy:** one cohort at a time, float32, del+gc after each

### RAM changes from v3.3
| Technique | Where |
|-----------|-------|
| `float32` cast on expression load | Cell 3 loader |
| `nlargest()` variance filter — no full sort | Cell 3 loader |
| `del data / model / opt` + `gc.collect()` after each cohort | Cell 5 |
| `workers=0` in DataLoader | Cell 5 |
| State dict stored `.cpu().clone()` only | Cell 5 |
| Sequential cohort loop — only one cohort in RAM at a time | Cell 5 |
| KM/figures generated and immediately closed per cohort | Cell 6 |
| Full model **not** kept in `COHORT_RESULTS` after export | Cell 7 |


### Cell 1 — Imports & Seeding

In [7]:
# ==============================================================================
# CELL 1: IMPORTS & SEEDING
# ==============================================================================
import os, random, warnings, platform, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # no GUI — saves RAM on headless machines
import matplotlib.pyplot as plt
import yaml, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore")
NB_VERSION = "3.3-ram"; PIPELINE_STAGE = "NB00"; SEED = 42

def enforce_determinism(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

enforce_determinism(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ {PIPELINE_STAGE} v{NB_VERSION} | Device: {device} | Seed: {SEED}")


✅ NB00 v3.3-ram | Device: cpu | Seed: 42


### Cell 2 — Config & Paths

In [8]:
# ==============================================================================
# CELL 2: CONFIG & PATHS
# ==============================================================================
CONFIG_PATH = Path("config.yaml")
if not CONFIG_PATH.exists(): raise FileNotFoundError(f"config.yaml missing from {Path.cwd()}")
with open(CONFIG_PATH, encoding="utf-8") as f: cfg = yaml.safe_load(f)
print(f"🚀 {cfg['project']['name']} v{cfg['project']['version']}")

BASE_DIR       = Path.cwd()
out_cfg        = cfg.get("output", {})
_ckpt_root     = out_cfg.get("checkpoint_dir", "checkpoints")
_results_root  = out_cfg.get("results_dir",    "results")
_figures_sub   = out_cfg.get("figures_subdir", "figures")
CHECKPOINT_DIR = BASE_DIR / _ckpt_root    / PIPELINE_STAGE
FIGURES_DIR    = BASE_DIR / _results_root / PIPELINE_STAGE / _figures_sub
RAW_DIR        = BASE_DIR / cfg.get("data", {}).get("raw_dir", "data/raw")
for _d in [CHECKPOINT_DIR, FIGURES_DIR]: _d.mkdir(parents=True, exist_ok=True)
print(f"   Checkpoints : {_ckpt_root}/{PIPELINE_STAGE}/")
print(f"   Figures     : {_results_root}/{PIPELINE_STAGE}/{_figures_sub}/")

pp = cfg.get("preprocessing",{}); mdl = cfg.get("model",{}); tr = cfg.get("training",{})
GLOBAL_HP = {
    "target_features": pp.get("variance_filter_top_n", 3000),
    "batch_size":  tr.get("batch_size",   64),
    "lr":          tr.get("lr",           5e-4),
    "weight_decay":tr.get("weight_decay", 0.01),
    "epochs":      tr.get("epochs",       40),
    "patience":    tr.get("patience",     8),
    "hidden_dim":  mdl.get("latent_dim",  64),
    "dropout_rate":mdl.get("dropout_rate",0.40),
    "cross_folds": mdl.get("cross_folds", 5),
}
COHORTS = []
if "cohorts" in cfg:
    for entry in cfg["cohorts"]:
        c = {**GLOBAL_HP}; c.update(entry.get("hparams", {}))
        c["name"]  = entry["name"]; c["label"] = entry.get("label", entry["name"])
        c["short"] = entry.get("short", entry["name"].replace("TCGA-","").lower())
        c["expression_file"] = RAW_DIR / entry["expression_file"]
        c["survival_file"]   = RAW_DIR / entry["survival_file"]
        COHORTS.append(c)
else:
    d = cfg.get("data",{}); c = {**GLOBAL_HP}
    c["name"] = cfg["project"].get("tcga_cohort","TCGA-UNK").replace("TCGA-","")
    c["label"] = cfg["project"].get("cancer_type",c["name"]).upper()
    c["expression_file"] = RAW_DIR / d.get("expression_file","TCGA-LGG.star_fpkm.tsv")
    c["survival_file"]   = RAW_DIR / d.get("survival_file","TCGA-LGG.survival.tsv")
    COHORTS = [c]
print(f"   Cohorts : {[c['short'] for c in COHORTS]}")
for c in COHORTS:
    print(f"   {c['name']:6s} | expr {'✅' if c['expression_file'].exists() else '❌ MISSING'}"
          f"  surv {'✅' if c['survival_file'].exists() else '❌ MISSING'}")


🚀 universal-survival-engine v3.3
   Checkpoints : checkpoints/NB00/
   Figures     : results/NB00/figures/
   Cohorts : ['lgg', 'kirc', 'luad']
   TCGA-LGG | expr ✅  surv ✅
   TCGA-KIRC | expr ✅  surv ✅
   TCGA-LUAD | expr ✅  surv ✅


### Cell 3 — RAM-Optimised Data Loader

In [9]:
# ==============================================================================
# CELL 3: RAM-OPTIMISED DATA LOADER
# ==============================================================================
# RAM management (from KEEP_NB04 template)
# • float32 everywhere — halves expression matrix vs float64
# • del + gc.collect() after every large object
# • torch.cuda.empty_cache() per fold
# • Sequential cohort processing — only one cohort in RAM at a time
# • workers=0 in DataLoader — no subprocess fork RAM copies
# • State dict stored .cpu().clone() — no GPU residual
def _trim(s):
    s = str(s).strip().replace(".", "-").upper()
    return "-".join(s.split("-")[:3]) if s.startswith("TCGA") else s

def _strip_v(ensg): return str(ensg).split(".")[0]

def load_cohort(name, expr_path, surv_path, top_n=3000, label=""):
    """RAM-optimised TCGA loader (float32, nlargest variance, early drop)."""
    print(f"  📂 {name}")

    # ── Survival — read only needed columns ──────────────────────────────────
    clin = pd.read_csv(surv_path, sep="\t", index_col=0)
    clin.index = pd.Index([_trim(i) for i in clin.index])
    clin = clin[~clin.index.duplicated(keep="first")]
    time_col  = next((c for c in ["OS.time","survival_time","time"] if c in clin.columns), None)
    event_col = next((c for c in ["OS","event","status"] if c in clin.columns), None)
    if not time_col or not event_col:
        raise KeyError(f"[{name}] No time/event columns in {list(clin.columns)}")

    # ── Expression — float32 on load, variance filter immediately ────────────
    df = pd.read_csv(expr_path, sep="\t", index_col=0)
    if df.shape[0] > df.shape[1]: df = df.T
    df.columns = pd.Index([_strip_v(c) for c in df.columns])
    df = df.loc[:, ~df.columns.duplicated(keep="first")]
    df.index = pd.Index([_trim(i) for i in df.index])
    df = df[~df.index.duplicated(keep="first")].astype(np.float32)  # float32 immediately

    # Variance filter via nlargest — never sorts full array
    top_genes = df.var(axis=0, ddof=0).nlargest(min(top_n, df.shape[1])).index.tolist()
    df = df[top_genes]  # shrink width before any further ops

    # ── Align, filter, clean ─────────────────────────────────────────────────
    common = df.index.intersection(clin.index)
    df     = df.loc[common]
    surv   = clin.loc[common, [time_col, event_col]].copy()
    surv.columns = ["t", "e"]
    valid  = surv["t"].notna() & surv["e"].notna() & (surv["t"] > 0)
    df     = df.loc[valid].dropna(axis=1).astype(np.float32)
    surv   = surv.loc[valid].astype(np.float32)

    # Cast survival to float32
    y_time  = surv["t"].values.astype(np.float32)
    y_event = surv["e"].values.astype(np.float32)

    ram_mb = df.memory_usage(deep=True).sum() / 1e6
    n_drop = int((~valid).sum())
    print(f"     {len(df)} patients | {df.shape[1]:,} genes | "
          f"{int(y_event.sum())} events ({100*y_event.mean():.1f}%) | "
          f"RAM: {ram_mb:.1f} MB" + (f" | dropped {n_drop} neg-time" if n_drop else ""))

    return {"name": name, "label": label or name,
            "X_raw": df.values.astype(np.float32),
            "gene_cols": df.columns.tolist(),
            "y_time": y_time, "y_event": y_event}

print("✅ load_cohort() defined.")


✅ load_cohort() defined.


### Cell 4 — Model & Loss

In [10]:
# ==============================================================================
# CELL 4: DATASET, MODEL & LOSS
# ==============================================================================
class SurvivalDataset(Dataset):
    def __init__(self, X, times, events):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.times=torch.tensor(times,dtype=torch.float32)
        self.events=torch.tensor(events,dtype=torch.float32)
    def __len__(self): return len(self.times)
    def __getitem__(self, i): return self.X[i], self.times[i], self.events[i]

class BaselineMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),    nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//2), nn.BatchNorm1d(hidden_dim//2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, 1, bias=False),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def cox_partial_loss(risks, times, events):
    order = torch.argsort(times, descending=True)
    risks_s = risks[order]; events_s = events[order]
    log_risk_cum = torch.logcumsumexp(risks_s, dim=0)
    nll = -(risks_s - log_risk_cum) * events_s
    return nll.sum() / events_s.sum().clamp(min=1.0)

def bootstrap_cindex(times, events, risks, n_boot=1000, seed=SEED):
    ci = concordance_index(times, -risks, events)
    rng = np.random.default_rng(seed); boot = []
    for _ in range(n_boot):
        bi = rng.choice(len(times), size=len(times), replace=True)
        if events[bi].sum() == 0: continue
        try: boot.append(concordance_index(times[bi], -risks[bi], events[bi]))
        except: continue
    lo, hi = (np.percentile(boot,[2.5,97.5]) if boot else (np.nan,np.nan))
    return ci, lo, hi, (np.std(boot) if boot else np.nan)

print("✅ Model, loss, bootstrap defined.")


✅ Model, loss, bootstrap defined.


### Cell 5 — Leakage-Free CV + Evaluation + Export (per cohort)

In [11]:
# ==============================================================================
# CELL 5: PER-COHORT CV + HONEST EVALUATION + CHECKPOINT
# RAM: one cohort loaded, trained, evaluated, saved, then deleted before next
# ==============================================================================
ALL_RESULTS = {}   # lightweight summary only — no tensors kept

for cohort_cfg in COHORTS:
    cname = cohort_cfg["name"]; sh = cohort_cfg["short"]
    print(f"\n{'='*68}\n🏥  {cname}  ({cohort_cfg['label']})\n{'='*68}")

    # ── Load (float32, nlargest, dropna) ─────────────────────────────────────
    data = load_cohort(cname,
                       cohort_cfg["expression_file"], cohort_cfg["survival_file"],
                       top_n=cohort_cfg["target_features"], label=cohort_cfg["label"])
    X_raw   = data["X_raw"]; y_time = data["y_time"]; y_event = data["y_event"]

    TARGET_FEATURES = cohort_cfg["target_features"]
    BATCH_SIZE = cohort_cfg["batch_size"]; LR = cohort_cfg["lr"]
    WEIGHT_DECAY = cohort_cfg["weight_decay"]; EPOCHS = cohort_cfg["epochs"]
    PATIENCE = cohort_cfg["patience"]; HIDDEN_DIM = cohort_cfg["hidden_dim"]
    DROPOUT_RATE = cohort_cfg["dropout_rate"]; CROSS_FOLDS = cohort_cfg["cross_folds"]

    kf = KFold(n_splits=CROSS_FOLDS, shuffle=True, random_state=SEED)
    fold_c_idx, fold_losses = [], []
    best_val_c = -1.; best_state = best_mask = best_scaler = val_idx_best = None

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_raw)):
        print(f"  🌀 Fold {fold+1}/{CROSS_FOLDS}")
        X_tr_r = X_raw[tr_idx]; X_vr = X_raw[val_idx]
        t_tr = y_time[tr_idx];  t_val = y_time[val_idx]
        e_tr = y_event[tr_idx]; e_val = y_event[val_idx]

        top_mask = np.argsort(X_tr_r.var(axis=0))[-TARGET_FEATURES:]
        scaler   = StandardScaler()
        X_tr     = scaler.fit_transform(X_tr_r[:, top_mask])
        X_val    = scaler.transform(X_vr[:, top_mask])
        del X_tr_r, X_vr   # free slices immediately

        train_ldr = DataLoader(SurvivalDataset(X_tr, t_tr, e_tr),
                               batch_size=BATCH_SIZE, shuffle=True,
                               drop_last=False, num_workers=0)  # workers=0 saves RAM

        enforce_determinism(SEED)
        model = BaselineMLP(TARGET_FEATURES, HIDDEN_DIM, DROPOUT_RATE).to(device)
        opt   = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        best_fc = -1.; best_fst = None; wait = 0; ep_losses = []

        for epoch in range(1, EPOCHS + 1):
            model.train(); ep_loss = 0.; n = 0
            for bx, bt, be in train_ldr:
                bx,bt,be = bx.to(device),bt.to(device),be.to(device)
                opt.zero_grad()
                loss = cox_partial_loss(model(bx), bt, be)
                loss.backward(); opt.step()
                ep_loss += loss.item() * bx.size(0); n += bx.size(0)
            avg = ep_loss / n; ep_losses.append(avg)
            model.eval()
            with torch.no_grad():
                val_risk = model(torch.tensor(X_val,dtype=torch.float32).to(device)).cpu().numpy()
            val_c = concordance_index(t_val, -val_risk, e_val)
            if epoch % 10 == 0 or epoch == 1:
                print(f"     Ep {epoch:3d} | Loss {avg:.4f} | Val C {val_c:.4f}")
            if val_c > best_fc:
                best_fc  = val_c
                # cpu().clone() — no GPU residual kept
                best_fst = {k: v.cpu().clone() for k,v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= PATIENCE: print(f"     🛑 Early stop ep {epoch}"); break

        fold_c_idx.append(best_fc); fold_losses.append(ep_losses)
        print(f"     ↳ Fold {fold+1} val C: {best_fc:.4f}")
        if best_fc > best_val_c:
            best_val_c, best_state = best_fc, best_fst
            best_mask, best_scaler, val_idx_best = top_mask.copy(), scaler, val_idx.copy()

        # ── Free fold objects immediately ─────────────────────────────────────
        del model, opt, X_tr, X_val
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()

    print(f"\n  📊 [{cname}] CV: {np.mean(fold_c_idx):.4f} ± {np.std(fold_c_idx):.4f}")

    # ── Honest val-fold evaluation ────────────────────────────────────────────
    best_model = BaselineMLP(TARGET_FEATURES, HIDDEN_DIM, DROPOUT_RATE).to(device)
    best_model.load_state_dict(best_state); best_model.eval()
    X_all = best_scaler.transform(X_raw[:, best_mask])
    with torch.no_grad():
        all_risks = best_model(torch.tensor(X_all,dtype=torch.float32).to(device)).cpu().numpy()
    del X_all

    vt, ve, vr = y_time[val_idx_best], y_event[val_idx_best], all_risks[val_idx_best]
    ci_v, lo_v, hi_v, sd_v = bootstrap_cindex(vt, ve, vr)
    ci_f, lo_f, hi_f, _    = bootstrap_cindex(y_time, y_event, all_risks)
    print(f"  ✅ HONEST val C: {ci_v:.4f}  [{lo_v:.4f}–{hi_v:.4f}]  SD±{sd_v:.4f}")
    print(f"  ⚠️  IN-SAMPLE  : {ci_f:.4f}  (do NOT report)")

    # ── Figures ───────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    for i, losses in enumerate(fold_losses):
        ax.plot(range(1,len(losses)+1), losses, alpha=0.7, lw=1.5, label=f"Fold {i+1}")
    ax.set_title(f"NB00 [{cname}] CV Loss", fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Cox NLL")
    ax.legend(fontsize=7, ncol=CROSS_FOLDS); ax.grid(True, ls=":", alpha=0.5)
    fig.tight_layout(); fig.savefig(FIGURES_DIR / f"NB00_{sh}_cv_loss.png", dpi=150, bbox_inches="tight")
    plt.close(fig)   # close immediately — don't accumulate figure RAM

    thresh = np.median(vr); hi_m = vr >= thresh; lo_m = ~hi_m
    lr_res = logrank_test(vt[hi_m], vt[lo_m], ve[hi_m], ve[lo_m])
    pval   = f"p={lr_res.p_value:.4f}" if lr_res.p_value >= 0.0001 else "p<0.0001"
    fig, ax = plt.subplots(figsize=(8, 5))
    for mask, lbl, col in [(hi_m,f"High (n={hi_m.sum()})","#DC143C"),
                            (lo_m,f"Low  (n={lo_m.sum()})","#008080")]:
        if mask.sum() > 1:
            KaplanMeierFitter().fit(vt[mask],ve[mask],label=lbl).plot_survival_function(
                ax=ax, ci_show=True, color=col, linewidth=2.5)
    ax.set_title(f"NB00 [{cname}] KM  C={ci_v:.4f} [{lo_v:.4f}–{hi_v:.4f}]  {pval}",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Days"); ax.set_ylabel("Survival"); ax.set_ylim(0,1.05)
    ax.legend(loc="lower left"); ax.grid(True, ls=":", alpha=0.4)
    fig.tight_layout(); fig.savefig(FIGURES_DIR / f"NB00_{sh}_km.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓ results/NB00/figures/NB00_{sh}_cv_loss.png + km.png")

    # ── Checkpoint ────────────────────────────────────────────────────────────
    ckpt_name = f"NB00_{sh}_v{NB_VERSION.replace('.','_')}_mlp_baseline.pt"
    torch.save({
        "pipeline_stage": PIPELINE_STAGE, "nb_version": NB_VERSION, "cohort": cname,
        "model_state_dict": best_state,
        "config": {"input_dim": TARGET_FEATURES, "hidden_dim": HIDDEN_DIM, "dropout": DROPOUT_RATE},
        "gene_mask": best_mask,
        "gene_cols": [data["gene_cols"][i] for i in best_mask],
        "scaler_mean": best_scaler.mean_, "scaler_var": best_scaler.var_,
        "cv_c_indices": fold_c_idx, "best_val_c": best_val_c,
        "honest_ci":  {"c_index":ci_v,"lo":lo_v,"hi":hi_v,"sd":sd_v,
                       "note":"val-fold only — honest hold-out"},
        "insample_ci":{"c_index":ci_f,"note":"full-cohort optimistic — do NOT report"},
    }, CHECKPOINT_DIR / ckpt_name)
    probe = torch.load(CHECKPOINT_DIR / ckpt_name, weights_only=False)
    assert probe["cohort"] == cname
    print(f"  💾 checkpoints/NB00/{ckpt_name}  ✅")

    # ── Store lightweight summary only — NO tensors / arrays ─────────────────
    ALL_RESULTS[sh] = {
        "mean_c": float(np.mean(fold_c_idx)), "std_c": float(np.std(fold_c_idx)),
        "ci_val": ci_v, "ci_val_lo": lo_v, "ci_val_hi": hi_v,
        "cohort": cname,
    }

    # ── Free ALL cohort RAM before next iteration ─────────────────────────────
    del data, X_raw, y_time, y_event, best_model, best_state, best_mask, best_scaler
    del all_risks, vt, ve, vr, fold_c_idx, fold_losses
    gc.collect()
    print(f"  🧹 [{cname}] RAM freed.")

print(f"\n{'='*68}\n  NB00 v{NB_VERSION} — HANDOFF SUMMARY")
print(f"  {'Cohort':<7} {'CV Mean±Std':>16}  {'Honest CI':>9}  [95% CI]")
print(f"  {'─'*55}")
for sh, r in ALL_RESULTS.items():
    print(f"  {sh:<7} {r['mean_c']:.4f} ± {r['std_c']:.4f}   {r['ci_val']:.4f}   "
          f"[{r['ci_val_lo']:.4f}–{r['ci_val_hi']:.4f}]")
    print(f"          └─ checkpoints/NB00/NB00_{sh}_v{NB_VERSION.replace('.','_')}_mlp_baseline.pt")
print(f"{'='*68}")



🏥  TCGA-LGG  (Brain LGG)
  📂 TCGA-LGG
     512 patients | 3,000 genes | 126 events (24.6%) | RAM: 6.2 MB | dropped 4 neg-time
  🌀 Fold 1/5
     Ep   1 | Loss 2.9190 | Val C 0.8035
     Ep  10 | Loss 2.3160 | Val C 0.8279
     🛑 Early stop ep 16
     ↳ Fold 1 val C: 0.8337
  🌀 Fold 2/5
     Ep   1 | Loss 2.6774 | Val C 0.8397
     Ep  10 | Loss 2.0469 | Val C 0.8606
     🛑 Early stop ep 19
     ↳ Fold 2 val C: 0.8652
  🌀 Fold 3/5
     Ep   1 | Loss 2.7345 | Val C 0.7599
     Ep  10 | Loss 2.1537 | Val C 0.8399
     Ep  20 | Loss 1.7117 | Val C 0.8303
     🛑 Early stop ep 22
     ↳ Fold 3 val C: 0.8507
  🌀 Fold 4/5
     Ep   1 | Loss 2.7386 | Val C 0.8326
     Ep  10 | Loss 2.1164 | Val C 0.8264
     🛑 Early stop ep 13
     ↳ Fold 4 val C: 0.8465
  🌀 Fold 5/5
     Ep   1 | Loss 2.6737 | Val C 0.6957
     Ep  10 | Loss 1.9415 | Val C 0.7698
     Ep  20 | Loss 1.6817 | Val C 0.7626
     🛑 Early stop ep 25
     ↳ Fold 5 val C: 0.7950

  📊 [TCGA-LGG] CV: 0.8382 ± 0.0238
  ✅ HONEST val C: 0.

In [12]:
# ==============================================================================
# CELL 5b: DEAD TOLL + SURVIVAL COUNT + KM MILESTONE TABLE  (added v3.3-ram+)
# Requires: ALL_RESULTS already populated, checkpoints saved.
# Loads each checkpoint to re-derive risk scores for full-cohort toll reporting.
# ==============================================================================
from lifelines import KaplanMeierFitter
import pandas as pd

MILESTONES_DAYS = [365, 3*365, 5*365]   # Year 1, 3, 5

print(f"\n{'='*72}")
print("  DEAD TOLL & SURVIVAL TABLE — NB00 MLP Baseline")
print(f"{'='*72}")

for cohort_cfg in COHORTS:
    cname  = cohort_cfg["name"]
    sh     = cohort_cfg["short"]
    ckpt_p = CHECKPOINT_DIR / f"NB00_{sh}_v{NB_VERSION.replace('.','_')}_mlp_baseline.pt"
    if not ckpt_p.exists():
        print(f"  [{cname}] checkpoint not found — skipping"); continue

    # ── A. Load raw event array from original data ────────────────────────────
    data    = load_cohort(cname, cohort_cfg["expression_file"],
                          cohort_cfg["survival_file"],
                          top_n=cohort_cfg["target_features"],
                          label=cohort_cfg["label"])
    y_time  = data["y_time"]
    y_event = data["y_event"]

    dead_toll      = int(y_event.sum())
    survival_count = int((1 - y_event).sum())
    n_total        = len(y_event)
    print(f"\n  [{cname}]  N={n_total}")
    print(f"    Dead Toll      (Σ δᵢ = 1) : {dead_toll}  ({100*dead_toll/n_total:.1f}%)")
    print(f"    Survival Count (Σ 1-δᵢ)   : {survival_count}  ({100*survival_count/n_total:.1f}%)")

    # ── B. Re-derive risk scores from saved checkpoint ────────────────────────
    ck     = torch.load(ckpt_p, weights_only=False)
    X_raw  = data["X_raw"]
    mask   = ck["gene_mask"]
    sc_m   = ck["scaler_mean"]; sc_v = ck["scaler_var"]
    X_sc   = (X_raw[:, mask] - sc_m) / np.sqrt(sc_v + 1e-8)

    # Reconstruct model from checkpoint config — not global vars
    cfg_r   = ck["config"]
    model_r = BaselineMLP(cfg_r["input_dim"], cfg_r["hidden_dim"],
                          cfg_r["dropout"]).to(device)
    model_r.load_state_dict(ck["model_state_dict"]); model_r.eval()
    with torch.no_grad():
        risk = model_r(torch.tensor(X_sc, dtype=torch.float32).to(device)).cpu().numpy().ravel()
    del model_r, X_sc; gc.collect()

    # ── C. Risk stratify at median ────────────────────────────────────────────
    thresh = np.median(risk)
    hi_m   = risk >= thresh
    lo_m   = ~hi_m

    # ── D. KM milestone table ─────────────────────────────────────────────────
    rows = []
    for group_label, mask_g in [("High-Risk", hi_m), ("Low-Risk", lo_m), ("Overall", np.ones(len(risk), dtype=bool))]:
        kmf = KaplanMeierFitter(label=group_label)
        kmf.fit(y_time[mask_g], y_event[mask_g])
        et  = kmf.event_table  # index = time points
        for ms in MILESTONES_DAYS:
            yr  = ms // 365
            # at_risk and events up to milestone
            sub = et[et.index <= ms]
            at_risk   = int(sub["at_risk"].iloc[-1])  if len(sub) else int(mask_g.sum())
            n_events  = int(sub["observed"].sum())
            n_censored= int(sub["censored"].sum())
            sf_val    = float(kmf.survival_function_at_times([ms]).iloc[0])
            rows.append({"Cohort": cname, "Group": group_label,
                         "Year": f"Y{yr}",
                         "At-Risk": at_risk,
                         "Events (Dead)": n_events,
                         "Censored": n_censored,
                         "S(t)": round(sf_val, 4)})

    tbl = pd.DataFrame(rows)
    print()
    print(tbl.to_string(index=False))

    del data, X_raw, risk; gc.collect()

print(f"\n{'='*72}")



  DEAD TOLL & SURVIVAL TABLE — NB00 MLP Baseline
  📂 TCGA-LGG
     512 patients | 3,000 genes | 126 events (24.6%) | RAM: 6.2 MB | dropped 4 neg-time

  [TCGA-LGG]  N=512
    Dead Toll      (Σ δᵢ = 1) : 126  (24.6%)
    Survival Count (Σ 1-δᵢ)   : 386  (75.4%)

  Cohort     Group Year  At-Risk  Events (Dead)  Censored   S(t)
TCGA-LGG High-Risk   Y1      186             26        46 0.8832
TCGA-LGG High-Risk   Y3       57             74       126 0.5529
TCGA-LGG High-Risk   Y5       19             97       141 0.2868
TCGA-LGG  Low-Risk   Y1      212              0        45 1.0000
TCGA-LGG  Low-Risk   Y3      103              1       153 0.9913
TCGA-LGG  Low-Risk   Y5       49              1       207 0.9913
TCGA-LGG   Overall   Y1      397             26        91 0.9419
TCGA-LGG   Overall   Y3      159             75       279 0.7771
TCGA-LGG   Overall   Y5       67             98       348 0.6245
  📂 TCGA-KIRC
     531 patients | 3,000 genes | 175 events (33.0%) | RAM: 6.4 MB

  [TC

In [13]:
# ==============================================================================
# CELL 5c: EMBEDDING EXPORT  (added — mirrors NB01/NB02/NB03)
# Requires: ALL_RESULTS populated, checkpoints saved (Cell 5 complete).
# Saves per-cohort latent embeddings for downstream audit and visualisation.
# NB00 BaselineMLP has no explicit encoder, so the 64-dim penultimate layer
# is used as the latent representation (layer before final Linear(64,1)).
# ==============================================================================
out_cfg_emb   = cfg.get("output", {})
EMBEDDINGS_DIR = BASE_DIR / out_cfg_emb.get("embeddings_dir", "embeddings") / PIPELINE_STAGE
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
print(f"   Embeddings  : {out_cfg_emb.get('embeddings_dir','embeddings')}/{PIPELINE_STAGE}/")

# Hook to extract 64-dim penultimate activation from BaselineMLP
def _extract_latent(model, X_tensor):
    """Run forward pass and capture output of the LayerNorm(64) layer."""
    latent = {}
    def _hook(module, inp, out):
        latent["z"] = out.detach().cpu()
    # BaselineMLP.net[-2] is LayerNorm(64) — the layer before Linear(64,1)
    handle = model.net[-2].register_forward_hook(_hook)
    with torch.no_grad():
        model(X_tensor)
    handle.remove()
    return latent["z"].numpy()

z_cols = [f"z_{i}" for i in range(64)]

for cohort_cfg in COHORTS:
    cname = cohort_cfg["name"]; sh = cohort_cfg["short"]
    ckpt_p = CHECKPOINT_DIR / f"NB00_{sh}_v{NB_VERSION.replace('.','_')}_mlp_baseline.pt"
    if not ckpt_p.exists():
        print(f"  [{cname}] checkpoint not found — skipping"); continue

    data    = load_cohort(cname, cohort_cfg["expression_file"],
                          cohort_cfg["survival_file"],
                          top_n=cohort_cfg["target_features"],
                          label=cohort_cfg["label"])
    X_raw   = data["X_raw"]; y_time = data["y_time"]; y_event = data["y_event"]

    ck      = torch.load(ckpt_p, weights_only=False)
    cfg_r   = ck["config"]; mask = ck["gene_mask"]
    sc_m    = ck["scaler_mean"]; sc_v = ck["scaler_var"]
    model_r = BaselineMLP(cfg_r["input_dim"], cfg_r["hidden_dim"],
                          cfg_r["dropout"]).to(device)
    model_r.load_state_dict(ck["model_state_dict"]); model_r.eval()

    # — held-out: checkpoint scaler (train-split only — leakage-safe) —
    X_ho     = (X_raw[:, mask] - sc_m) / np.sqrt(sc_v + 1e-8)
    X_ho_t   = torch.tensor(X_ho, dtype=torch.float32).to(device)
    z_ho     = _extract_latent(model_r, X_ho_t)
    with torch.no_grad():
        risk_ho = model_r(X_ho_t).cpu().numpy().ravel()
    df_ho = pd.DataFrame(z_ho, columns=z_cols)
    df_ho["risk_score"] = risk_ho
    df_ho["survival_time"] = y_time; df_ho["event"] = y_event
    ho_path = EMBEDDINGS_DIR / f"NB00_{sh}_heldout_latents.csv"
    df_ho.to_csv(ho_path, index=False)
    print(f"  📁 NB00_{sh}_heldout_latents.csv  (leakage-safe — checkpoint train scaler)")
    del X_ho, X_ho_t, z_ho, risk_ho, df_ho

    # — full-cohort: refit scaler on all patients (representation only) —
    from sklearn.preprocessing import StandardScaler
    sc_full  = StandardScaler()
    top_mask = ck["gene_mask"]
    X_fc     = sc_full.fit_transform(X_raw[:, top_mask]).astype(np.float32)
    X_fc_t   = torch.tensor(X_fc, dtype=torch.float32).to(device)
    z_fc     = _extract_latent(model_r, X_fc_t)
    with torch.no_grad():
        risk_fc = model_r(X_fc_t).cpu().numpy().ravel()
    del X_fc, X_fc_t
    df_fc = pd.DataFrame(z_fc, columns=z_cols)
    df_fc["risk_score"] = risk_fc
    df_fc["survival_time"] = y_time; df_fc["event"] = y_event
    fc_path = EMBEDDINGS_DIR / f"NB00_{sh}_fullcohort_latents.csv"
    df_fc.to_csv(fc_path, index=False)
    print(f"  📁 NB00_{sh}_fullcohort_latents.csv  (full-cohort scaler — representation only)")
    del z_fc, risk_fc, df_fc, sc_full

    del model_r, data, X_raw, y_time, y_event; gc.collect()
    print(f"  🧹 [{cname}] embeddings done.")

print(f"\n  Embeddings saved to embeddings/{PIPELINE_STAGE}/")


   Embeddings  : embeddings/NB00/
  📂 TCGA-LGG
     512 patients | 3,000 genes | 126 events (24.6%) | RAM: 6.2 MB | dropped 4 neg-time


ValueError: Shape of passed values is (512, 32), indices imply (512, 64)